In [2]:
!pip install sentence-transformers rank-bm25

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Documents
documents = [
    "Neural networks are used in deep learning",
    "Deep learning models use gradient descent",
    "Today's weather is sunny"
]

# Convert documents to embeddings
doc_embeddings = model.encode(documents)

# User query
query = "deep learning"

# Convert query to embedding
query_embedding = model.encode(query)

# Compute similarity
scores = np.dot(doc_embeddings, query_embedding)

# Get best match
best_idx = np.argmax(scores)

print("Query:", query)
print("Best Match:", documents[best_idx])
print("Score:", scores[best_idx])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: deep learning
Best Match: Neural networks are used in deep learning
Score: 0.6858681


In [6]:
def calculate_mrr(results, relevant_docs):
    """
    Calculates Mean Reciprocal Rank (MRR).

    Args:
        results (list): A list of ranked document indices or identifiers.
        relevant_docs (list): A list of relevant document indices or identifiers.

    Returns:
        float: The MRR score.
    """
    for i, doc_idx in enumerate(results):
        if doc_idx in relevant_docs:
            return 1.0 / (i + 1)
    return 0.0

def calculate_ndcg(results, relevant_docs, k=None):
    """
    Calculates Normalized Discounted Cumulative Gain (NDCG).

    Args:
        results (list): A list of ranked document indices or identifiers.
        relevant_docs (list): A list of relevant document indices or identifiers.
        k (int, optional): The number of top results to consider. If None, considers all results.

    Returns:
        float: The NDCG score.
    """
    if k is None:
        k = len(results)

    dcg = 0.0
    for i in range(min(k, len(results))):
        doc_idx = results[i]
        relevance = 1 if doc_idx in relevant_docs else 0
        dcg += (2**relevance - 1) / (np.log2(i + 2))

    # Calculate Ideal DCG
    ideal_relevance = [1] * len(relevant_docs) + [0] * (len(results) - len(relevant_docs))
    ideal_relevance.sort(reverse=True)
    idcg = 0.0
    for i in range(min(k, len(ideal_relevance))):
        idcg += (2**ideal_relevance[i] - 1) / (np.log2(i + 2))

    if idcg == 0:
        return 0.0
    return dcg / idcg

In [7]:
# For our query "deep learning", documents 0 and 1 are relevant.
# documents = [
#     "Neural networks are used in deep learning",  # Index 0
#     "Deep learning models use gradient descent",   # Index 1
#     "Today's weather is sunny"                     # Index 2
# ]
relevant_doc_indices = [0, 1]

# Get BM25 scores (already calculated in M4KkI7FJMht-)
# From the output of M4KkI7FJMht-:
# Score: 0.17 -> Doc 0
# Score: 0.18 -> Doc 1
# Score: 0.00 -> Doc 2

bm25_scores = bm25.get_scores(tokenized_query)

# Rank documents based on BM25 scores (higher score is better)
bm25_ranked_indices = np.argsort(bm25_scores)[::-1] # Get indices in descending order of scores

# Convert to a list of original document indices
bm25_ranked_doc_indices = bm25_ranked_indices.tolist()

print("BM25 Ranked Document Indices:", bm25_ranked_doc_indices)

# Calculate MRR for BM25
bm25_mrr = calculate_mrr(bm25_ranked_doc_indices, relevant_doc_indices)
print(f"BM25 MRR: {bm25_mrr:.2f}")

# Calculate NDCG for BM25 (e.g., k=3)
bm25_ndcg = calculate_ndcg(bm25_ranked_doc_indices, relevant_doc_indices, k=3)
print(f"BM25 NDCG@3: {bm25_ndcg:.2f}")

BM25 Ranked Document Indices: [1, 0, 2]
BM25 MRR: 1.00
BM25 NDCG@3: 1.00


In [8]:
# Get Sentence Transformer scores
# scores = np.dot(doc_embeddings, query_embedding)

# Rank documents based on Sentence Transformer scores (higher score is better)
sentence_transformer_ranked_indices = np.argsort(scores)[::-1] # Get indices in descending order of scores

# Convert to a list of original document indices
sentence_transformer_ranked_doc_indices = sentence_transformer_ranked_indices.tolist()

print("Sentence Transformer Ranked Document Indices:", sentence_transformer_ranked_doc_indices)

# Calculate MRR for Sentence Transformer
sentence_transformer_mrr = calculate_mrr(sentence_transformer_ranked_doc_indices, relevant_doc_indices)
print(f"Sentence Transformer MRR: {sentence_transformer_mrr:.2f}")

# Calculate NDCG for Sentence Transformer (e.g., k=3)
sentence_transformer_ndcg = calculate_ndcg(sentence_transformer_ranked_doc_indices, relevant_doc_indices, k=3)
print(f"Sentence Transformer NDCG@3: {sentence_transformer_ndcg:.2f}")

Sentence Transformer Ranked Document Indices: [0, 1, 2]
Sentence Transformer MRR: 1.00
Sentence Transformer NDCG@3: 1.00


In [1]:
!pip install -q langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.4 MB/s eta 0:00:00


In [3]:
from getpass import getpass

groq_api_key = getpass("Enter your Groq API Key: ")

from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)

# Prompt
zero_shot_summary_prompt = PromptTemplate(
    input_variables=["earnings_text"],
    template="""
Summarize the following earnings call snippet.

Requirements:
- Maximum 3 bullet points
- Mention revenue trend
- Mention profitability
- Mention future outlook

Text:
{earnings_text}
"""
)

# Sample text
earnings_text = """
Revenue increased 18% year-over-year driven by strong cloud adoption.
Net profit margin improved from 12% to 16%.
Management expects continued growth next quarter and plans to expand AI offerings.
"""

# Create prompt
formatted_prompt = zero_shot_summary_prompt.format(
    earnings_text=earnings_text
)

# Invoke model
response = llm.invoke(formatted_prompt)

print(response.content)

Enter your Groq API Key: ··········
Here is a summary of the earnings call snippet in 3 bullet points:
* The company's revenue trend is positive, with an 18% year-over-year increase driven by strong cloud adoption.
* In terms of profitability, the net profit margin has improved from 12% to 16%, indicating a significant enhancement in the company's bottom line.
* Looking ahead, management has a favorable future outlook, expecting continued growth in the next quarter and planning to expand its AI offerings to further drive business success.


In [5]:
from getpass import getpass
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Enter Groq API Key
groq_api_key = getpass("Enter Groq API Key: ")

# Initialize LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)

# Few-Shot Prompt
few_shot_summary_prompt = PromptTemplate(
    input_variables=["earnings_text"],
    template="""
Example 1

Input:
Revenue increased 15% YoY. Net income rose 10%.
Management expects strong demand next quarter.

Output:
• Revenue grew 15%
• Profitability improved
• Positive future outlook

Example 2

Input:
Revenue declined 5%. Operating costs increased.
Management announced restructuring plans.

Output:
• Revenue decreased
• Costs increased
• Restructuring planned

Now summarize:

Input:
{earnings_text}

Output:
"""
)

# Test Data
earnings_text = """
Revenue increased 22% year-over-year driven by cloud services.
Net profit margin improved from 14% to 19%.
Management expects continued growth and expansion into new markets.
"""

# Format Prompt
prompt = few_shot_summary_prompt.format(
    earnings_text=earnings_text
)

# Generate Response
response = llm.invoke(prompt)

print(response.content)

Enter Groq API Key: ··········
• Revenue grew 22%
• Profitability improved
• Positive future outlook


In [7]:
from getpass import getpass
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Enter Groq API Key
groq_api_key = getpass("Enter Groq API Key: ")

# Initialize LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)

# Prompt
ticket_classifier_prompt = PromptTemplate(
    input_variables=["ticket"],
    template="""
You are a customer support classifier.

Possible Classes:

1. Billing
2. Tech
3. Refund
4. General
5. Escalate

Think step-by-step:

Step 1: Identify customer issue
Step 2: Determine customer intent
Step 3: Select most appropriate class

Ticket:
{ticket}

Return:

Reasoning:
Class:
"""
)

# Sample ticket
ticket = """
I was charged twice for my subscription this month.
Please reverse the extra charge immediately.
"""

# Format prompt
prompt = ticket_classifier_prompt.format(ticket=ticket)

# Invoke model
response = llm.invoke(prompt)

print(response.content)

Enter Groq API Key: ··········
Reasoning:
To classify this ticket, let's break it down step by step:

Step 1: Identify customer issue - The customer has been charged twice for their subscription this month, resulting in an extra, unwanted charge.

Step 2: Determine customer intent - The customer intends to have the extra charge reversed as soon as possible, indicating a need for a correction in their billing.

Step 3: Select most appropriate class - Given the customer's issue and intent, the most relevant class for this ticket is related to correcting a billing error.

Class: Billing


In [10]:
from getpass import getpass
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Enter Groq API Key
groq_api_key = getpass("Enter Groq API Key: ")

# Initialize Model
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)

# Prompt
generate_snippet_prompt = PromptTemplate(
    input_variables=["company", "quarter"],
    template="""
You are a financial analyst.

Generate a realistic earnings call snippet for {company} for {quarter}.

Include:
- Revenue growth or decline
- Profitability metrics
- Future guidance
- Market conditions

Return 100-150 words.
"""
)

# Create Prompt
prompt = generate_snippet_prompt.format(
    company="NexaBank",
    quarter="Q2 2026"
)

# Generate Response
response = llm.invoke(prompt)

print(response.content)

Enter Groq API Key: ··········
"NexaBank's Q2 2026 results showed revenue growth of 8% year-over-year, driven by strong loan demand. Net income rose 12% to $250 million, with a return on equity of 15.6%. Our efficiency ratio improved to 55%, reflecting ongoing cost management efforts. Looking ahead, we expect 6-8% revenue growth for the full year, with net income expected to rise 10-12%. Despite market uncertainty, our diversified portfolio and solid capital position position us well for future growth. We're cautiously optimistic about the economy, but remain vigilant in managing risk and maintaining our underwriting standards."


In [11]:
!pip install -q rouge-score

  Preparing metadata (setup.py) ... done


In [12]:
from rouge_score import rouge_scorer

reference_summary = (
    "Revenue increased and management provided positive guidance."
)

generated_summary = (
    "Revenue grew and outlook remains positive."
)

scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

score = scorer.score(
    reference_summary,
    generated_summary
)

print("ROUGE-L Precision:",
      score['rougeL'].precision)

print("ROUGE-L Recall:",
      score['rougeL'].recall)

print("ROUGE-L F1:",
      score['rougeL'].fmeasure)

ROUGE-L Precision: 0.5
ROUGE-L Recall: 0.42857142857142855
ROUGE-L F1: 0.4615384615384615
